In [26]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('nyu-mll/glue', 'mrpc')
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [27]:
train_sample_set = dataset["train"][:20]

for i in range(len(train_sample_set["sentence1"])):
    print(f"Sample {i+1}:")
    print(f"Sentence 1: {train_sample_set['sentence1'][i]}")
    print(f"Sentence 2: {train_sample_set['sentence2'][i]}")
    print(f"Label: {train_sample_set['label'][i]}")
    print(f"ID: {train_sample_set['idx'][i]}")
    print()

Sample 1:
Sentence 1: Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
Sentence 2: Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
Label: 1
ID: 0

Sample 2:
Sentence 1: Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
Sentence 2: Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .
Label: 0
ID: 1

Sample 3:
Sentence 1: They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
Sentence 2: On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .
Label: 1
ID: 2

Sample 4:
Sentence 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Sentence 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing 

In [28]:
full_train_sample_set = dataset["train"][:]

for i in range(len(full_train_sample_set["sentence1"])):
    if full_train_sample_set["sentence1"][i] == full_train_sample_set["sentence2"][i]:
        print(f"Sample {i+1}:")
        print(f"Sentence 1: {full_train_sample_set['sentence1'][i]}")
        print(f"Sentence 2: {full_train_sample_set['sentence2'][i]}")
        print(f"Label: {full_train_sample_set['label'][i]}")
        print(f"ID: {full_train_sample_set['idx'][i]}")
        print()

In [29]:
full_test_sample_set = dataset["test"][:]

full_valid_sample_set = dataset["validation"][:]

In [30]:
train_sentences = full_train_sample_set["sentence1"] + full_train_sample_set["sentence2"]

test_sentences = full_test_sample_set["sentence1"] + full_test_sample_set["sentence2"]

valid_sentences = full_valid_sample_set["sentence1"] + full_valid_sample_set["sentence2"]

train_set = set(train_sentences)
valid_set = set(valid_sentences)
test_set = set(test_sentences)

train_valid_overlap = train_set.intersection(valid_set)
train_test_overlap = train_set.intersection(test_set)
valid_test_overlap = valid_set.intersection(test_set)

print(len(train_valid_overlap))
print(len(train_test_overlap))
print(len(valid_test_overlap))

43
244
24


In [31]:
duplicate_dict = {}

for sentence in train_sentences:
    if sentence in test_sentences or sentence in valid_sentences:
        if sentence in duplicate_dict:
            duplicate_dict[sentence] += 1
        else:
            duplicate_dict[sentence] = 1

for i in duplicate_dict.items():
    print(i)    

('The dollar was at 117.85 yen against the Japanese currency , up 0.1 percent .', 1)
('Shares of Hartford rose $ 2.88 to $ 46.50 in New York Stock Exchange composite trading .', 1)
("Standard & Poor 's 500 stock index futures declined 4.40 points to 983.50 , while Nasdaq futures fell 6.5 points to 1,206.50 .", 1)
('For example , Anhalt said , children typically treat a 20-ounce soda bottle as one serving , although it actually contains 2 ½ 8-ounce servings .', 1)
('The pressure will intensify today , with national coverage of the final round planned by ESPN and words that are even more difficult .', 1)
('" This deal makes sense for both companies , " Halla said in a prepared statement .', 2)
("A few thousand troops , most from the division 's 3rd Brigade Combat Team based at Fort Benning in Columbus , began returning last week , with flights continuing through Friday .", 1)
('On Thursday , a Post article argued that a 50 basis point cut from the Fed was more likely .', 1)
("The acquisi

In [32]:
duplicate_df = pd.DataFrame(
    duplicate_dict.items(),
    columns=["sentence", "count"]
)

duplicate_df.sort_values("count", ascending=False)

,sentence,count
89,Physicians who perform the procedure would fac...,3
141,The technology-laced Nasdaq Composite Index wa...,3
35,Kerry Kennedy Cuomo and Andrew Cuomo have been...,2
41,"Wal- Mart , Kohl 's Corp. , Family Dollar Stor...",2
68,Federal investigators are looking for possible...,2
...,...,...
23,Legato stockholders will get 0.9 of a share of...,1
24,Myanmar 's pro-democracy leader Aung San Suu K...,1
25,The company 's chief executive retired and chi...,1
26,The study was presented yesterday in Chicago a...,1


Potential label ambiguity: ID 17 is labeled non-paraphrase even though the two sentences express almost the same main event. Sentence 1 contains an additional clause about the “dolphin scheme,” which may explain the negative label, but the core meaning is highly similar.

Pre-tokenization spacing artifacts: punctuation, quotation marks, possessives, and currency symbols are separated by spaces, e.g. " the witness ", Dominick 's, and $ 2.5 billion.

Numerical/factual disagreement inside positive pairs — Sample 5 has different dollar amounts and percentages but is still labeled paraphrase.

Entity/reference differences — one sentence may name an entity while the other uses a more generic reference.

Repeated sentences